# SpaceX — SQL exploratory analysis (historical dashboard sample)

This is **not** IBM’s original SQL lab dataset. The following SQL was executed against the included 56-launch historical sample. The column `Class` represents first-stage landing outcome, not mission outcome.

In [1]:
import pandas as pd, sqlite3
from pathlib import Path
csv = Path('../../data/historical_dash_56.csv')
if not csv.exists():
    csv = Path('data/historical_dash_56.csv')
df = pd.read_csv(csv)
connection = sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL', connection, index=False)
print('Loaded', len(df), 'launches')

Loaded 56 launches


## 01 Sites

In [2]:
query = 'SELECT LaunchSite, COUNT(*) AS launches FROM SPACEXTBL GROUP BY LaunchSite ORDER BY launches DESC'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

  LaunchSite  launches
 CCAFS LC-40        26
  KSC LC-39A        13
 VAFB SLC-4E        10
CCAFS SLC-40         7


## 02 Site Success

In [3]:
query = 'SELECT LaunchSite, COUNT(*) AS n, SUM(Class) AS successful, ROUND(100.0*AVG(Class),1) AS success_pct FROM SPACEXTBL GROUP BY LaunchSite ORDER BY success_pct DESC'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

  LaunchSite  n  successful  success_pct
  KSC LC-39A 13          10         76.9
CCAFS SLC-40  7           3         42.9
 VAFB SLC-4E 10           4         40.0
 CCAFS LC-40 26           7         26.9


## 03 Payload

In [4]:
query = 'SELECT BoosterCategory, COUNT(*) AS flights, ROUND(AVG(PayloadMass),0) AS avg_payload_kg, ROUND(MAX(PayloadMass),0) AS max_payload_kg FROM SPACEXTBL GROUP BY BoosterCategory ORDER BY flights DESC'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

BoosterCategory  flights  avg_payload_kg  max_payload_kg
             FT       24          4502.0          9600.0
           v1.1       15          2535.0          4707.0
             B4       11          5058.0          9600.0
           v1.0        5           340.0           677.0
             B5        1          3600.0          3600.0


## 04 Success Counts

In [5]:
query = 'SELECT Class, COUNT(*) AS launches FROM SPACEXTBL GROUP BY Class ORDER BY Class DESC'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

 Class  launches
     1        24
     0        32


## 05 Top Payload

In [6]:
query = 'SELECT FlightNumber, LaunchSite, PayloadMass, Class FROM SPACEXTBL ORDER BY PayloadMass DESC LIMIT 5'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

 FlightNumber  LaunchSite  PayloadMass  Class
           29 VAFB SLC-4E       9600.0      1
           37 VAFB SLC-4E       9600.0      1
           42 VAFB SLC-4E       9600.0      1
           46 VAFB SLC-4E       9600.0      0
           51 VAFB SLC-4E       9600.0      0


## 06 Ccafs

In [7]:
query = "SELECT FlightNumber, LaunchSite, Class FROM SPACEXTBL WHERE LaunchSite LIKE 'CCA%' ORDER BY FlightNumber LIMIT 5"
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

 FlightNumber  LaunchSite  Class
            1 CCAFS LC-40      0
            2 CCAFS LC-40      0
            3 CCAFS LC-40      0
            4 CCAFS LC-40      0
            5 CCAFS LC-40      0


## 07 Booster Rank

In [8]:
query = 'SELECT BoosterCategory, COUNT(*) AS flights, SUM(Class) AS successful, ROUND(100.0*AVG(Class),1) AS success_pct FROM SPACEXTBL GROUP BY BoosterCategory ORDER BY success_pct DESC'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

BoosterCategory  flights  successful  success_pct
             B5        1           1        100.0
             FT       24          16         66.7
             B4       11           6         54.5
           v1.1       15           1          6.7
           v1.0        5           0          0.0


## 08 Payload Range

In [9]:
query = "SELECT CASE WHEN PayloadMass < 2000 THEN '<2,000 kg' WHEN PayloadMass < 5000 THEN '2,000–4,999 kg' ELSE '5,000+ kg' END AS payload_band, COUNT(*) AS flights, ROUND(100.0*AVG(Class),1) AS success_pct FROM SPACEXTBL GROUP BY payload_band ORDER BY flights DESC"
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

  payload_band  flights  success_pct
2,000–4,999 kg       29         55.2
     5,000+ kg       14         35.7
     <2,000 kg       13         23.1


## 09 Max Payload

In [10]:
query = 'SELECT FlightNumber, LaunchSite, BoosterCategory, PayloadMass FROM SPACEXTBL WHERE PayloadMass=(SELECT MAX(PayloadMass) FROM SPACEXTBL)'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

 FlightNumber  LaunchSite BoosterCategory  PayloadMass
           29 VAFB SLC-4E              FT       9600.0
           37 VAFB SLC-4E              FT       9600.0
           42 VAFB SLC-4E              B4       9600.0
           46 VAFB SLC-4E              FT       9600.0
           51 VAFB SLC-4E              B4       9600.0


## 10 Ranking

In [11]:
query = 'SELECT LaunchSite, COUNT(*) AS flights, SUM(Class) AS successes FROM SPACEXTBL GROUP BY LaunchSite ORDER BY successes DESC'
result = pd.read_sql_query(query, connection)
print(result.to_string(index=False))

  LaunchSite  flights  successes
  KSC LC-39A       13         10
 CCAFS LC-40       26          7
 VAFB SLC-4E       10          4
CCAFS SLC-40        7          3
